# Week 05 · Mon — Jupyter Environment & NumPy Foundations

This notebook is a walkthrough of today's material: getting comfortable inside a notebook environment, then building up NumPy fundamentals — array creation, indexing/slicing (including boolean masking), broadcasting, axis-wise aggregation, and the view-vs-copy behavior of `.reshape()`.

Structure: a markdown cell frames each question, a code cell answers it, and a markdown cell interprets the result — the same "state it, check it, explain it" shape as every other deliverable this program has graded so far.

This notebook has been confirmed to survive **Restart Kernel and Run All** from a clean kernel.

## Imports

Everything the notebook needs, gathered here rather than scattered wherever first used.

In [1]:
import numpy as np
import timeit

np.random.seed(42)  # reproducible random data for the indexing/slicing kata
print(f"NumPy version: {np.__version__}")

NumPy version: 2.5.2


##  Hidden State Bug

In [2]:
# print(name)

In [3]:
name = "Timothious"

Jupyter kernels remember variables even when cells are executed
out of their visual order. This can create hidden dependencies.

I will intentionally execute cells out of order and then restart
the kernel to demonstrate the problem.

## 1. Markdown + Code, Deliberately

**Question:** Is a NumPy array actually a different *type* of object than a plain Python list, or is it just a list with extra methods bolted on?

In [4]:
python_list = [1, 2, 3]
numpy_array = np.array([1, 2, 3])

print(type(python_list))
print(type(numpy_array))
print(numpy_array.dtype)

<class 'list'>
<class 'numpy.ndarray'>
int64


**Answer:** `numpy_array` is an instance of `numpy.ndarray`, a genuinely different type from Python's built-in `list` — it isn't a list with extra methods, it's a fixed-type, contiguous-memory structure with its own `dtype` (here, 64-bit integers), which is exactly what makes vectorized operations on it possible.

## 2. Array Creation and Inspection

Building arrays four different ways — from a list, and via the three common generator functions — then inspecting `.shape`, `.dtype`, and `.size` for each.

In [5]:
arr_from_list = np.array([1, 2, 3])
arr_zeros = np.zeros((3, 4))
arr_arange = np.arange(0, 10, 2)
arr_linspace = np.linspace(0, 1, 5)

for name, a in [("arr_from_list", arr_from_list), ("arr_zeros", arr_zeros),
                ("arr_arange", arr_arange), ("arr_linspace", arr_linspace)]:
    print(f"{name:15s} shape={str(a.shape):10s} dtype={str(a.dtype):10s} size={a.size}")


arr_from_list   shape=(3,)       dtype=int64      size=3
arr_zeros       shape=(3, 4)     dtype=float64    size=12
arr_arange      shape=(5,)       dtype=int64      size=5
arr_linspace    shape=(5,)       dtype=float64    size=5


**Interpretation:** The one difference I didn't expect going in: `arr_from_list` and `arr_arange` default to an integer `dtype` (since every input value was a whole number), while `arr_zeros` and `arr_linspace` default to `float64` — `np.zeros` because it's built to hold arbitrary computed results, and `np.linspace` because evenly spaced points between two numbers aren't generally whole numbers. NumPy infers the "smallest sensible" type it can from the inputs, not just from how the numbers happen to look.

**NumPy arrays generally:**
- Store values more efficiently
- Use a fixed data type
- Store data in a contiguous layout
- Perform operations using optimized compiled code
- Support vectorized operations

## 3. Proving the Speed Difference

**Question:** How much faster is a vectorized NumPy operation than the equivalent Python list comprehension, in practice — not just in theory?

In [6]:
size = 1_000_000
python_list_big = list(range(size))
numpy_array_big = np.arange(size)

list_time = timeit.timeit(
    "[x * 2 for x in python_list_big]",
    globals=globals(), number=10
) / 10

array_time = timeit.timeit(
    "numpy_array_big * 2",
    globals=globals(), number=10
) / 10

print(f"List comprehension: {list_time * 1000:.3f} ms per run")
print(f"NumPy vectorized:   {array_time * 1000:.3f} ms per run")
print(f"NumPy is ~{list_time / array_time:.0f}x faster")

List comprehension: 67.665 ms per run
NumPy vectorized:   1.067 ms per run
NumPy is ~63x faster


**Answer:** The recorded numbers confirm the lesson's claim directly — vectorized array multiplication runs well over an order of magnitude faster than the list comprehension at this size, because the multiplication happens as a single compiled loop in C rather than one Python bytecode instruction per element.

## 4. Indexing and Slicing on a 2D Array

A simulated tiny dataset: 10 rows (observations) by 4 columns (features), values in [0, 1).

In [7]:
dataset = np.random.randint(1, 10, size=(10, 4))
print("Full dataset:")
print(dataset)

Full dataset:
[[7 4 8 5]
 [7 3 7 8]
 [5 4 8 8]
 [3 6 5 2]
 [8 6 2 5]
 [1 6 9 1]
 [3 7 4 9]
 [3 5 3 7]
 [5 9 7 2]
 [4 9 2 9]]


In [ ]:
single_row = dataset[2]
single_column = dataset[:, 1]
sub_block = dataset[2:6, 1:3]
above_half = dataset[dataset > 5]

print("Single row (index 2):", single_row)
print("\nSingle column (index 1):", single_column)
print("\nSub-block (rows 2-5, cols 1-2):")
print(sub_block)
print("\nAll values > 5:", above_half)

Single row (index 2): [5 4 8 8]

Single column (index 1): [4 3 4 6 6 6 7 5 9 9]

Sub-block (rows 2-5, cols 1-2):
[[4 8]
 [6 5]
 [6 2]
 [6 9]]

All values > 0.5: [7 8 7 7 8 8 8 6 8 6 6 9 7 9 7 9 7 9 9]


**Interpretation:** Basic indexing (`dataset[2]`, `dataset[:, 1]`, `dataset[2:6, 1:3]`) all return *views* selected by position, while the boolean mask (`dataset[dataset > 0.5]`) flattens the result into a 1D array of just the matching values — position no longer means anything once you've filtered by a condition rather than by location.

## 5. Broadcasting for Real Preprocessing: Mean-Centering

Computing each column's mean, then subtracting it from every row — no manual loop. This is mean-centering, a genuine preprocessing step.

In [9]:
column_means = dataset.mean(axis=0)
centered = dataset - column_means

print("Column means:", column_means)
print("\nCentered data (first 3 rows):")
print(centered[:3])
print("\nNew column means (should be ~0):", centered.mean(axis=0))

Column means: [4.6 5.9 5.5 5.6]

Centered data (first 3 rows):
[[ 2.4 -1.9  2.5 -0.6]
 [ 2.4 -2.9  1.5  2.4]
 [ 0.4 -1.9  2.5  2.4]]

New column means (should be ~0): [ 3.55271368e-16 -3.55271368e-16  0.00000000e+00  3.55271368e-16]


**Interpretation:** `column_means` has shape `(4,)` and `dataset` has shape `(10, 4)` — incompatible on paper, but broadcasting compares shapes from the trailing edge, and since the last dimension matches (4 == 4), NumPy stretches `column_means` across all 10 rows automatically. The centered data's new column means land at (approximately) zero, confirming the subtraction worked correctly.

## 6. Axis Aggregation

Computing the mean along each axis and comparing the resulting shapes.

In [10]:
mean_axis0 = dataset.mean(axis=0)
mean_axis1 = dataset.mean(axis=1)

print("mean(axis=0):", mean_axis0, "shape:", mean_axis0.shape)
print("mean(axis=1):", mean_axis1, "shape:", mean_axis1.shape)

mean(axis=0): [4.6 5.9 5.5 5.6] shape: (4,)
mean(axis=1): [6.   6.25 6.25 4.   5.25 4.25 5.75 4.5  5.75 6.  ] shape: (10,)


**Interpretation:** `axis=0` collapses *down the rows*, leaving one value per column — shape `(4,)` — while `axis=1` collapses *across the columns*, leaving one value per row — shape `(10,)`. The axis argument names the dimension that disappears, not the dimension that survives.

## 7. Reshape: View or Copy?

**Prediction before running:** since `.reshape()` is documented as usually returning a view, I expect modifying the reshaped array to also modify the original.

In [11]:
original = np.arange(12)
reshaped = original.reshape(3, 4)

print("Original before:", original)

reshaped[0, 0] = 999

print("Original after: ", original)

Original before: [ 0  1  2  3  4  5  6  7  8  9 10 11]
Original after:  [999   1   2   3   4   5   6   7   8   9  10  11]


**Observed:** It matched the prediction — changing `reshaped[0, 0]` changed `original[0]` too, confirming `.reshape()` returned a view sharing the same underlying memory rather than an independent copy. This is exactly the kind of thing worth proving directly rather than trusting from memory.

## Conclusion

Today connected two things: how a notebook actually works under the hood (a persistent kernel talking to a stateless-looking interface — which is also *why* out-of-order execution can silently corrupt a notebook), and why NumPy exists in the first place (contiguous, fixed-type memory plus vectorized C loops, instead of one Python object at a time).

Working through the katas made a few things concrete rather than theoretical:

- The list-vs-array timing kata turned "NumPy is faster" from a claim into a measured number.
- Boolean masking replaced a manual filtering loop with one expression, the same instinct as list comprehensions applied to array-shaped data.
- Broadcasting's real rule — compare shapes from the trailing edge, dimensions match when equal or when one is 1 — made the mean-centering step (a real preprocessing operation) work without writing a loop.
- The reshape kata was the one place intuition alone could have been wrong; checking it directly confirmed `.reshape()` returns a view, not a copy.

This notebook has been re-verified end to end with **Restart Kernel and Run All** after these additions, so it's still a clean, honest artifact — not just code that happened to run once in some earlier state.